In [1]:
!pip install datasets
!pip install pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 9.1 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.5.1+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.5.1+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system =

In [2]:
import numpy as np
import pandas as pd

from google.colab import drive
from typing import List
from datasets import Dataset, ClassLabel, DatasetDict

In [3]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
df = pd.read_csv('./drive/MyDrive/questoes/base_de_conhecimento/questoes.csv')
df.head()

,titulo_do_texto,texto,descritor,comando,resposta_item,resposta,opcoes,ano,tipo,origem
0,Entenda como se forma um arco-íris!,O arco colorido surge quando os raios de luz d...,D23 - Identificar os níveis de linguagem e/ou ...,A linguagem utilizada no trecho “Quando observ...,c,formal.,"['científica.', 'coloquial.\r', 'formal.', 're...",6° ano,SPAECE,Avaliação paraná
1,Tomar banho depois de comer faz mal?,"Na verdade, não é indicado tomar banho de imer...",D23 - Identificar os níveis de linguagem e/ou ...,No trecho “... diminuir os pulsos e daí você s...,d,se fala no dia a dia.,"['é falada em uma região do país. ', 'é utiliz...",6° ano,SPAECE,Avaliação paraná
2,Blog da Galera: Minha aventura para ver o show...,"Isabella Vieira, da Galera CAPRICHO, foi atrás...",D23 - Identificar os níveis de linguagem e/ou ...,"Nesse texto, no trecho “E aí, galera!? Aqui qu...",a,conversas entre amigos.,"['conversas entre amigos.\r', 'discursos polít...",6° ano,SPAECE,Avaliação paraná
3,Lagosta “algodão­doce”: o que explica essa col...,"Um grupo de pescadores dos EUA1 , teve uma enc...",D23 - Identificar os níveis de linguagem e/ou ...,"Nesse texto, o trecho: “No entanto, biólogos m...",d,formal,"['coloquial.', 'técnica.', 'antiga.', 'formal']",6° ano,SPAECE,Avaliação paraná
4,-,"Beto Oi! Meu nome é Beto e vivo na Zoolândia, ...",D23 - Identificar os níveis de linguagem e/ou ...,"Nesse texto, no trecho “... vivo na Zoolândia,...",b,informal.,"['científica.', 'informal.', 'padrão.', 'regio...",6° ano,SPAECE,Avaliação paraná


In [6]:
df = df.dropna()
df.isnull().values.any()

False

In [7]:
descriptor_table = df['descritor'].value_counts().reset_index()
descriptor_table.columns = ['descritor', 'quantidade']
descriptor_table

,descritor,quantidade
0,D06 - Distinguir fato de opinião relativa ao f...,38
1,D23 - Identificar os níveis de linguagem e/ou ...,33
2,D05 - Identificar o tema ou assunto de um texto.,33
3,D20 - Reconhecer o efeito de sentido decorrent...,30
4,D01 - Localizar informações explícitas em um t...,27
5,D19 – Reconhecer o efeito de sentido decorrent...,25
6,D17 - Reconhecer o sentido das relações lógico...,23
7,D11 - Reconhecer os elementos que compõem uma ...,22
8,D21 - Reconhecer o efeito decorrente do empreg...,20
9,D07 - Diferenciar a informação principal das s...,20


In [8]:
dataset = Dataset.from_pandas(df)

dataset = dataset.map(lambda example: {"descritor_codigo": example["descritor"].split(" - ")[0]})
unique_descritores_codigos = list(set(dataset['descritor_codigo']))
dataset = dataset.cast_column('descritor_codigo', ClassLabel(names=unique_descritores_codigos))

split = dataset.train_test_split(test_size=0.1, stratify_by_column='descritor_codigo', seed=42)

dataset_dict = DatasetDict({
    'train': split['train'],
    'validation': split['test']
})

split_train = dataset_dict['train'].train_test_split(test_size=0.1, stratify_by_column='descritor_codigo', seed=42)

dataset_dict['train'] = split_train['train']
dataset_dict['test'] = split_train['test']

dataset_dict['train'].to_csv('dataset_treino.csv', index=False)
dataset_dict['validation'].to_csv('dataset_validacao.csv', index=False)
dataset_dict['test'].to_csv('dataset_test.csv', index=False)

Map:   0%|          | 0/324 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/324 [00:00<?, ? examples/s]

Creating CSV from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating CSV from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating CSV from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

48996